# LeetCode 1556 — Thousand Separator bằng Python

**Link đề:** https://leetcode.com/problems/thousand-separator/
**Độ khó:** Easy · **Chủ đề:** String

```python
class Solution:
    def thousandSeparator(self, n: int) -> str:
        ...
```

### Mục tiêu học
1. Hiểu quy tắc nhóm **3 chữ số tính từ phải sang** (giống cách viết `1.234.567` ở Việt Nam).
2. Thành thạo: `str(n)`, duyệt ngược `reversed`, slicing âm `s[-3:]`, `"." .join(...)`.
3. Hiểu cách làm **bằng toán**: `divmod(n, 1000)` và **định dạng đệm số 0** `f"{x:03d}"` (vì sao `1.005` không được viết thành `1.5`).
4. Biết "đường tắt" của Python `f"{n:,}"` và khi nào nên/không nên dùng nó trong phỏng vấn.
5. Trace, chứng minh, Time/Space, test đối chiếu 5 cách, bài tập biến thể (dấu phân cách tùy ý, nhóm kiểu Ấn Độ, bỏ dấu).

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài — thêm dấu chấm phân cách hàng nghìn

Cho số nguyên `n`, chèn dấu chấm `"."` làm **dấu phân cách hàng nghìn** và trả về kết quả dạng **chuỗi**.

| Ví dụ | `n` | Output |
|---|---|---|
| 1 | `987` | `"987"` |
| 2 | `1234` | `"1.234"` |
| thêm | `123456789` | `"123.456.789"` |
| thêm | `0` | `"0"` |
| thêm | `1000005` | `"1.000.005"` |

**Constraints:** `0 <= n <= 2^31 - 1` (tối đa `2147483647`, tức 10 chữ số).

Quy tắc: bắt đầu từ **chữ số cuối**, cứ đủ 3 chữ số thì chèn một dấu chấm ở bên trái — trừ khi đã hết số (không có dấu chấm ở đầu chuỗi).

Câu hỏi gợi mở:
- Vì sao phải đếm **từ phải**? Thử nhóm từ trái với `1234`: được `"123.4"` — sai!
- `1000005` có nhóm giữa là `000` — nếu làm bằng toán (`% 1000`) ta sẽ nhận được số `0`, làm sao in ra `"000"`?

In [1]:
n = 1234567
s = str(n)
print("Chuỗi:", s, "| độ dài:", len(s))
print("Nhóm từ trái (SAI) :", ".".join(s[i:i + 3] for i in range(0, len(s), 3)))
print("Mong muốn (từ phải): 1.234.567")

Chuỗi: 1234567 | độ dài: 7
Nhóm từ trái (SAI) : 123.456.7
Mong muốn (từ phải): 1.234.567


`range(0, len(s), 3)` tạo `0, 3, 6, ...` — tham số thứ ba là **bước nhảy** (step).

## 2. Nền tảng Python cần dùng

### 2.1. `str(n)` và độ dài chuỗi

`str(1234)` → `"1234"`. Số chữ số của `n` là `len(str(n))`. Đặc biệt `str(0) == "0"` (một chữ số).

In [2]:
for n in [0, 7, 987, 1234, 2**31 - 1]:
    print(n, "→", repr(str(n)), "| số chữ số:", len(str(n)))

0 → '0' | số chữ số: 1
7 → '7' | số chữ số: 1
987 → '987' | số chữ số: 3
1234 → '1234' | số chữ số: 4
2147483647 → '2147483647' | số chữ số: 10


`2**31` là lũy thừa (`**`): `2^31 = 2147483648`.

### 2.2. Slicing với chỉ số âm

Chỉ số âm đếm từ cuối: `s[-1]` là ký tự cuối, `s[-3:]` là **3 ký tự cuối**, `s[:-3]` là **mọi thứ trừ 3 ký tự cuối**. Nếu chuỗi ngắn hơn 3, `s[-3:]` trả cả chuỗi và `s[:-3]` trả chuỗi rỗng — không lỗi!

In [3]:
s = "1234567"
print(s[-3:], "|", s[:-3])
t = "12"
print(repr(t[-3:]), "|", repr(t[:-3]))

567 | 1234
'12' | ''


### 2.3. `reversed` và `[::-1]`

`reversed(s)` cho các ký tự theo thứ tự ngược (một iterator); `s[::-1]` tạo hẳn chuỗi đảo ngược. Cách "duyệt từ phải" sẽ dùng chúng.

In [4]:
s = "1234"
print(list(reversed(s)), s[::-1])

['4', '3', '2', '1'] 4321


### 2.4. `divmod` và đệm số 0 `f"{x:03d}"`

- `divmod(a, b)` trả về **cặp** `(a // b, a % b)`. Với `b = 1000`: `divmod(1234567, 1000) == (1234, 567)` — tách 3 chữ số cuối.
- `f"{x:03d}"`: in số nguyên (`d`), độ rộng 3, **đệm `0`** bên trái. `f"{5:03d}" == "005"`.

In [5]:
print(divmod(1234567, 1000))
print(divmod(1234, 1000))
for x in [5, 50, 500, 0]:
    print(x, "→", f"{x:03d}", "| str thường:", str(x))

(1234, 567)
(1, 234)
5 → 005 | str thường: 5
50 → 050 | str thường: 50
500 → 500 | str thường: 500
0 → 000 | str thường: 0


### 2.5. `sep.join(list)`

`".".join(["1", "234", "567"])` → `"1.234.567"`. Dấu chấm chỉ được chèn **giữa** các phần tử, không có ở đầu/cuối — đúng điều ta muốn.

In [6]:
print(".".join(["1", "234", "567"]))
print(".".join(["987"]))          # một phần tử → không có dấu chấm

1.234.567
987


## 3. Cách 1 — Duyệt từ phải, đếm 3 rồi chèn dấu (trực giác)

Làm như con người viết tay: đọc chữ số **từ phải**, ghi vào kết quả; mỗi khi đã ghi 3 chữ số **và còn chữ số phía trước** thì ghi thêm `"."`. Cuối cùng đảo ngược lại.

In [7]:
def sep_scan_back(n: int) -> str:
    s = str(n)
    out = []
    count = 0
    for i in range(len(s) - 1, -1, -1):     # i chạy từ cuối về 0
        out.append(s[i])
        count += 1
        if count % 3 == 0 and i > 0:        # đủ 3 và còn chữ số bên trái
            out.append(".")
    return "".join(reversed(out))

for n in [987, 1234, 123456789, 0, 1000005]:
    print(n, "→", sep_scan_back(n))

987 → 987
1234 → 1.234
123456789 → 123.456.789
0 → 0
1000005 → 1.000.005


`range(len(s) - 1, -1, -1)`: bắt đầu `len(s)-1`, dừng **trước** `-1` (tức chạy tới 0), bước `-1` — duyệt ngược chỉ số.

Điều kiện `i > 0` cực kỳ quan trọng: với `123456` (6 chữ số), sau khi ghi chữ số `1` (i = 0) count = 6 chia hết cho 3, nhưng đã hết số → **không** được thêm dấu chấm ở đầu.

## 4. Insight — nhóm đầu tiên có độ dài `len % 3`

Thay vì đếm từng chữ số, nhìn **cấu trúc**: tất cả các nhóm đều dài 3, **trừ nhóm đầu tiên** (bên trái) có thể ngắn hơn. Độ dài nhóm đầu:

```text
first = len(s) % 3     (nếu bằng 0 thì nhóm đầu dài 3)
```

Ví dụ `"1234567"`: `7 % 3 = 1` → nhóm đầu `"1"`, sau đó `"234"`, `"567"`.

In [8]:
for s in ["987", "1234", "12345", "123456", "1234567"]:
    first = len(s) % 3 or 3
    print(f"{s:>8}: len={len(s)}, nhóm đầu dài {first} → '{s[:first]}', phần còn lại '{s[first:]}'")

     987: len=3, nhóm đầu dài 3 → '987', phần còn lại ''
    1234: len=4, nhóm đầu dài 1 → '1', phần còn lại '234'
   12345: len=5, nhóm đầu dài 2 → '12', phần còn lại '345'
  123456: len=6, nhóm đầu dài 3 → '123', phần còn lại '456'
 1234567: len=7, nhóm đầu dài 1 → '1', phần còn lại '234567'


`len(s) % 3 or 3`: toán tử `or` trả về vế đầu nếu nó "đúng" (khác 0), ngược lại trả vế sau. Vậy khi `% 3 == 0` ta được `3`.

## 5. Lời giải chính — Cắt nhóm bằng slicing rồi `join`

### 5.1. Từng đoạn
1. `s = str(n)`.
2. `first = len(s) % 3 or 3` — độ dài nhóm đầu.
3. `groups = [s[:first]]`, sau đó thêm `s[i:i+3]` với `i = first, first+3, ...`.
4. `return ".".join(groups)`.

### 5.2. Ghép lại

In [9]:
def sep_groups(n: int) -> str:
    s = str(n)
    first = len(s) % 3 or 3
    groups = [s[:first]]
    for i in range(first, len(s), 3):
        groups.append(s[i:i + 3])
    return ".".join(groups)

for n in [987, 1234, 123456789, 0, 1000005, 2**31 - 1]:
    print(n, "→", sep_groups(n))

987 → 987
1234 → 1.234
123456789 → 123.456.789
0 → 0
1000005 → 1.000.005
2147483647 → 2.147.483.647


## 6. TRACE từng bước

`n = 1000005` → `s = "1000005"`, `len = 7`, `first = 7 % 3 = 1`:

| bước | i | đoạn cắt | groups |
|---|---|---|---|
| khởi tạo | — | `s[:1] = "1"` | `["1"]` |
| 1 | 1 | `s[1:4] = "000"` | `["1", "000"]` |
| 2 | 4 | `s[4:7] = "005"` | `["1", "000", "005"]` |
| join | | | `"1.000.005"` |

Nhờ làm trên **chuỗi**, các số 0 ở giữa được giữ nguyên tự nhiên.

In [10]:
def trace_groups(n):
    s = str(n)
    first = len(s) % 3 or 3
    groups = [s[:first]]
    print(f"s={s!r}, len={len(s)}, first={first}, groups={groups}")
    for step, i in enumerate(range(first, len(s), 3), start=1):
        groups.append(s[i:i + 3])
        print(f"bước {step}: i={i}, cắt s[{i}:{i+3}]={s[i:i+3]!r} → groups={groups}")
    ans = ".".join(groups)
    print("Kết quả:", ans)
    return ans

assert trace_groups(1000005) == "1.000.005"
print()
assert trace_groups(987) == "987"
print()
assert trace_groups(123456) == "123.456"

s='1000005', len=7, first=1, groups=['1']
bước 1: i=1, cắt s[1:4]='000' → groups=['1', '000']
bước 2: i=4, cắt s[4:7]='005' → groups=['1', '000', '005']
Kết quả: 1.000.005

s='987', len=3, first=3, groups=['987']
Kết quả: 987

s='123456', len=6, first=3, groups=['123']
bước 1: i=3, cắt s[3:6]='456' → groups=['123', '456']
Kết quả: 123.456


Với `987`, `range(3, 3, 3)` rỗng → không thêm nhóm nào → `"987"` không có dấu chấm ✓. Với `123456`, `first = 3` (vì `6 % 3 == 0`) ✓.

## 7. Các lời giải khác

### 7.1. Cách 3 — Cắt dần 3 ký tự cuối (slicing âm)

Lặp: lấy `s[-3:]` đưa vào đầu danh sách nhóm, rồi `s = s[:-3]`, đến khi `s` rỗng.

In [11]:
def sep_chop_tail(n: int) -> str:
    s = str(n)
    groups = []
    while s:
        groups.append(s[-3:])        # 3 ký tự cuối (hoặc ít hơn)
        s = s[:-3]                   # bỏ 3 ký tự cuối
    return ".".join(reversed(groups))

for n in [987, 1234, 1000005, 0]:
    print(n, "→", sep_chop_tail(n))

987 → 987
1234 → 1.234
1000005 → 1.000.005
0 → 0


`while s:` lặp khi chuỗi **khác rỗng** (chuỗi rỗng là "sai"). Ta `append` rồi đảo ngược ở cuối thay vì chèn vào đầu (`insert(0, ...)` tốn `O(số nhóm)` mỗi lần).

### 7.2. Cách 4 — Thuần toán học với `divmod` + đệm 0

Không dùng `str(n)` để cắt: lặp `n, r = divmod(n, 1000)`, nhóm `r` là 3 chữ số cuối. Mọi nhóm **trừ nhóm đầu** phải đệm đủ 3 chữ số (`f"{r:03d}"`).

In [12]:
def sep_math(n: int) -> str:
    if n < 1000:
        return str(n)
    groups = []
    while n >= 1000:
        n, r = divmod(n, 1000)
        groups.append(f"{r:03d}")     # nhóm giữa/cuối: đủ 3 chữ số
    groups.append(str(n))             # nhóm đầu: không đệm
    return ".".join(reversed(groups))

for n in [987, 1234, 1000005, 0, 1000]:
    print(n, "→", sep_math(n))

987 → 987
1234 → 1.234
1000005 → 1.000.005
0 → 0
1000 → 1.000


Trace cách toán với `n = 1000005`:

In [13]:
def trace_math(n):
    groups = []
    while n >= 1000:
        q, r = divmod(n, 1000)
        print(f"divmod({n}, 1000) = ({q}, {r}) → nhóm '{r:03d}'")
        groups.append(f"{r:03d}"); n = q
    print(f"nhóm đầu: '{n}'")
    groups.append(str(n))
    return ".".join(reversed(groups))

print(trace_math(1000005))

divmod(1000005, 1000) = (1000, 5) → nhóm '005'
divmod(1000, 1000) = (1, 0) → nhóm '000'
nhóm đầu: '1'
1.000.005


### 7.3. Cách 5 — Định dạng có sẵn của Python

Định dạng `f"{n:,}"` chèn dấu **phẩy** hàng nghìn kiểu Anh/Mỹ; ta chỉ cần thay `","` bằng `"."`. Ngoài ra `f"{n:_}"` dùng dấu gạch dưới.

In [14]:
def sep_builtin(n: int) -> str:
    return f"{n:,}".replace(",", ".")

print(f"{1234567:,}", "|", f"{1234567:_}")
for n in [987, 1234, 1000005, 0]:
    print(n, "→", sep_builtin(n))

1,234,567 | 1_234_567
987 → 987
1234 → 1.234
1000005 → 1.000.005
0 → 0


Cách này được LeetCode chấp nhận, ngắn nhất — nhưng trong phỏng vấn, người hỏi thường muốn thấy bạn **tự cài** (cách 2/3/4). Hãy biết cả hai.

### 7.4. Bảng so sánh

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Duyệt ngược đếm 3 | chèn `.` mỗi 3 chữ số | `O(d)` | `O(d)` | sát cách viết tay; cẩn thận `i > 0` |
| 2. **Nhóm đầu `len % 3`** | slicing + `join` | `O(d)` | `O(d)` | **nên dùng**, rõ ràng |
| 3. Cắt đuôi `s[-3:]` | lặp bỏ 3 ký tự cuối | `O(d²)` lý thuyết (mỗi `s[:-3]` tạo chuỗi mới), `d ≤ 10` nên không đáng kể | `O(d)` | ngắn gọn |
| 4. `divmod` + `03d` | thuần toán | `O(d)` | `O(d)` | luyện xử lý chữ số; nhớ đệm 0 |
| 5. `f"{n:,}"` | thư viện | `O(d)` | `O(d)` | nhanh khi đi làm thực tế |

Với `d` = số chữ số của `n` (≤ 10).

## 8. Code nộp LeetCode

In [15]:
class Solution:
    def thousandSeparator(self, n: int) -> str:
        s = str(n)
        first = len(s) % 3 or 3                 # độ dài nhóm đầu (1..3)
        groups = [s[:first]]
        for i in range(first, len(s), 3):
            groups.append(s[i:i + 3])
        return ".".join(groups)

sol = Solution()
print(sol.thousandSeparator(987))
print(sol.thousandSeparator(1234))

987
1.234


Phiên bản một dòng (cũng Accepted):

```python
class Solution:
    def thousandSeparator(self, n: int) -> str:
        return f"{n:,}".replace(",", ".")
```

## 9. Vì sao đúng?

Gọi `s = str(n)` có `d` chữ số, viết `d = 3q + r` với `r = d % 3`.

- Nếu `r > 0`: nhóm đầu dài `r`, theo sau là `q` nhóm dài 3. Nếu `r = 0`: nhóm đầu dài 3, theo sau `q - 1` nhóm dài 3. Trong cả hai trường hợp, **mọi nhóm sau nhóm đầu dài đúng 3** và chúng **kết thúc đúng tại chữ số cuối** (vì `first + 3·k = d`).
- Định nghĩa "phân cách hàng nghìn" chính là: các nhóm 3 chữ số đếm từ phải, nhóm trái cùng gồm phần dư (1..3 chữ số). Hai mô tả trùng nhau → kết quả đúng.
- `join` chỉ đặt dấu chấm **giữa** các nhóm → không có dấu chấm thừa ở đầu/cuối.

**Cách 4 (toán):** mỗi `divmod(n, 1000)` tách đúng 3 chữ số thấp nhất `r` (giá trị `0..999`). Đệm `03d` khôi phục các số 0 đứng đầu của nhóm đó (ví dụ `5 → "005"`). Nhóm cao nhất `n < 1000` không đệm vì số không có chữ số 0 vô nghĩa ở đầu.

## 10. Time / Space

Gọi `d` = số chữ số của `n` (`d ≤ 10` theo constraints, tổng quát `d = O(log₁₀ n)`).

- **Time:** `str(n)` `O(d)`, cắt nhóm tổng cộng `O(d)`, `join` `O(d)` → **`O(d)` = `O(log n)`**.
- **Space:** chuỗi kết quả dài `d + ⌊(d-1)/3⌋` → **`O(d)`**.

Vì `d ≤ 10`, thực tế mọi cách đều gần như tức thời.

## 11. Bộ test — mẫu đề, biên, phản ví dụ, đối chiếu

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual!r}")

sol = Solution()
check("đề 1", sol.thousandSeparator(987), "987")
check("đề 2", sol.thousandSeparator(1234), "1.234")
check("n = 0", sol.thousandSeparator(0), "0")
check("1 chữ số", sol.thousandSeparator(7), "7")
check("đúng 1000", sol.thousandSeparator(1000), "1.000")
check("999", sol.thousandSeparator(999), "999")
check("6 chữ số", sol.thousandSeparator(123456), "123.456")
check("0 ở giữa", sol.thousandSeparator(1000005), "1.000.005")
check("0 cuối", sol.thousandSeparator(51040), "51.040")
check("max int32", sol.thousandSeparator(2**31 - 1), "2.147.483.647")

PASS đề 1: '987'
PASS đề 2: '1.234'
PASS n = 0: '0'
PASS 1 chữ số: '7'
PASS đúng 1000: '1.000'
PASS 999: '999'
PASS 6 chữ số: '123.456'
PASS 0 ở giữa: '1.000.005'
PASS 0 cuối: '51.040'
PASS max int32: '2.147.483.647'


**Phản ví dụ cho cách toán quên đệm 0:**

In [17]:
def sep_math_no_pad(n):
    groups = []
    while n >= 1000:
        n, r = divmod(n, 1000)
        groups.append(str(r))          # SAI: mất số 0 đầu nhóm
    groups.append(str(n))
    return ".".join(reversed(groups))

for n in [1234, 1000005, 51040, 1000]:
    print(n, "| không đệm:", sep_math_no_pad(n), "| đúng:", sep_groups(n))

1234 | không đệm: 1.234 | đúng: 1.234
1000005 | không đệm: 1.0.5 | đúng: 1.000.005
51040 | không đệm: 51.40 | đúng: 51.040
1000 | không đệm: 1.0 | đúng: 1.000


`1234` tình cờ đúng (nhóm `234` không có 0 đầu), nhưng `1000005` thành `"1.0.5"` — sai hoàn toàn.

### 11.1. Đối chiếu 5 cách

Ta kiểm tra trên các số "đặc biệt" (lũy thừa của 10, ±1) và số ngẫu nhiên. Ngoài ra còn kiểm tra **tính chất**: bỏ dấu chấm phải ra lại `str(n)`, và mọi nhóm sau nhóm đầu dài đúng 3.

In [18]:
import random
special = [0, 1, 9, 10, 99, 100, 999, 1000, 1001, 9999, 10000, 99999, 100000, 999999, 1000000, 2**31 - 1]
special += [10**k + d for k in range(1, 10) for d in (-1, 0, 1)]
rng = random.Random(1556)
inputs = special + [rng.randint(0, 2**31 - 1) for _ in range(3000)]
for n in inputs:
    exp = sep_builtin(n)
    assert sep_scan_back(n) == exp
    assert sep_groups(n) == exp
    assert sep_chop_tail(n) == exp
    assert sep_math(n) == exp
    assert Solution().thousandSeparator(n) == exp
    parts = exp.split(".")
    assert "".join(parts) == str(n)
    assert 1 <= len(parts[0]) <= 3 and all(len(p) == 3 for p in parts[1:])
print(f"{len(inputs)} input: 5 cách khớp nhau, tính chất nhóm đúng")

3043 input: 5 cách khớp nhau, tính chất nhóm đúng


`exp.split(".")` tách chuỗi tại mỗi dấu chấm thành list các nhóm — phép ngược của `join`.

## 12. Lỗi người mới hay gặp

### Lỗi 1 — Nhóm từ trái sang
Đã thấy ở mục 1: `"123.4"` thay vì `"1.234"`.

### Lỗi 2 — Dấu chấm thừa ở đầu (quên điều kiện `i > 0`)

In [19]:
def sep_scan_back_bug(n):
    s = str(n); out = []; count = 0
    for i in range(len(s) - 1, -1, -1):
        out.append(s[i]); count += 1
        if count % 3 == 0:                   # SAI: thiếu "and i > 0"
            out.append(".")
    return "".join(reversed(out))

for n in [987, 123456, 1234]:
    print(n, "| lỗi:", sep_scan_back_bug(n), "| đúng:", sep_groups(n))

987 | lỗi: .987 | đúng: 987
123456 | lỗi: .123.456 | đúng: 123.456
1234 | lỗi: 1.234 | đúng: 1.234


Chỉ xuất hiện khi số chữ số **chia hết cho 3** — lại một lý do phải test `987`, `123456`.

### Lỗi 3 — Quên đệm 0 khi làm bằng toán
Đã thấy ở mục 11 (`1.0.5`). **Sửa:** `f"{r:03d}"` hoặc `str(r).zfill(3)`.

In [20]:
print("5".zfill(3), "42".zfill(3), "123".zfill(3))

005 042 123


`str.zfill(width)` đệm `0` bên trái cho đủ độ rộng — tương đương `f"{x:03d}"` nhưng làm trên chuỗi.

### Lỗi 4 — `n = 0` với cách toán dùng `while n > 0`
```python
while n > 0:
    n, r = divmod(n, 1000); groups.append(...)
```
Với `n = 0` vòng lặp không chạy → trả chuỗi **rỗng** thay vì `"0"`. **Sửa:** xử lý `n < 1000` riêng hoặc dùng điều kiện `n >= 1000` như mục 7.2.

In [21]:
def sep_math_zero_bug(n):
    groups = []
    while n > 0:
        n, r = divmod(n, 1000)
        groups.append(f"{r:03d}" if n > 0 else str(r))
    return ".".join(reversed(groups))

print(repr(sep_math_zero_bug(0)), "| đúng:", repr(sep_math(0)))
print(repr(sep_math_zero_bug(1000005)), "(các số khác vẫn đúng → lỗi rất dễ lọt!)")

'' | đúng: '0'
'1.000.005' (các số khác vẫn đúng → lỗi rất dễ lọt!)


### Lỗi 5 — Dùng `f"{n:,}"` nhưng quên `.replace`
Trả về `"1,234"` → Wrong Answer. Đề yêu cầu **dấu chấm**.

## 13. Mẹo nhận dạng pattern

- "Nhóm/định dạng từ **phải sang**" → nghĩ tới `len % k` cho nhóm đầu, hoặc duyệt ngược, hoặc `divmod(n, 10^k)`.
- Làm việc với chữ số bằng toán → luôn hỏi "**số 0 ở đầu nhóm** có bị mất không?" → đệm `0{k}d` / `zfill`.
- Ghép các phần có dấu phân cách → `sep.join(parts)` thay vì tự cộng chuỗi và xử lý dấu thừa.
- Biên cần test: `0`, số có số chữ số chia hết cho 3, số có nhóm `000`.
- Bài liên quan: 273 *Integer to English Words* (Hard — cũng tách theo nhóm 1000), 1694 *Reformat Phone Number*, 482 *License Key Formatting*.

## 14. Biến thể

### 14.1. Dấu phân cách và kích thước nhóm tùy ý

In [22]:
def group_digits(n: int, sep: str = ".", k: int = 3) -> str:
    s = str(n)
    first = len(s) % k or k
    return sep.join([s[:first]] + [s[i:i + k] for i in range(first, len(s), k)])

print(group_digits(1234567))
print(group_digits(1234567, sep=" "))
print(group_digits(12345678, sep="-", k=4))

1.234.567
1 234 567
1234-5678


### 14.2. Cách nhóm kiểu Ấn Độ: `12,34,567`
3 chữ số cuối thành một nhóm, phần còn lại nhóm **2** chữ số.

In [23]:
def indian_format(n: int) -> str:
    s = str(n)
    if len(s) <= 3:
        return s
    head, tail = s[:-3], s[-3:]
    first = len(head) % 2 or 2
    parts = [head[:first]] + [head[i:i + 2] for i in range(first, len(head), 2)]
    return ",".join(parts + [tail])

for n in [999, 1234, 1234567, 123456789]:
    print(n, "→", indian_format(n))

999 → 999
1234 → 1,234
1234567 → 12,34,567
123456789 → 12,34,56,789


### 14.3. Phép ngược: từ chuỗi có dấu về số

In [24]:
def parse_separated(t: str) -> int:
    return int(t.replace(".", ""))

print(parse_separated("1.000.005"), parse_separated("987"))

1000005 987


## 15. Bài tập (TODO không tự chạy)

### Bài 1 — Tính tay
Với `n = 20230101`: `len(str(n))` bằng mấy, `first` bằng mấy, các nhóm là gì, kết quả là gì?

### Bài 2 — Đoán output
```python
print(divmod(7005, 1000), f"{5:03d}", f"{7005:,}")
```

### Bài 3 — TODO: viết bản "nhóm đầu `len % 3`"

In [25]:
def exercise_thousand_separator(n: int) -> str:
    # TODO 1: s = str(n); first = len(s) % 3 or 3
    # TODO 2: groups = [s[:first]] + các đoạn 3 ký tự tiếp theo
    # TODO 3: ".".join(groups)
    pass

# assert exercise_thousand_separator(987) == "987"
# assert exercise_thousand_separator(1234) == "1.234"
# assert exercise_thousand_separator(1000005) == "1.000.005"

### Bài 4 — TODO: bản thuần toán (không dùng `str(n)` để cắt, chỉ dùng `divmod`)

In [26]:
def exercise_separator_math(n: int) -> str:
    # TODO: divmod(n, 1000) lặp; đệm 0 cho mọi nhóm trừ nhóm đầu; nhớ n = 0
    pass

# assert exercise_separator_math(0) == "0"
# assert exercise_separator_math(1000005) == "1.000.005"

### Bài 5 — TODO: định dạng số thập phân
Cho chuỗi số dương dạng `"1234567.891"`, chèn dấu chấm cho **phần nguyên**, dùng **dấu phẩy** làm dấu thập phân (kiểu Việt Nam): `"1.234.567,891"`. Không có phần thập phân thì giữ nguyên phần nguyên.

In [27]:
def exercise_vn_decimal(t: str) -> str:
    # TODO: tách t theo "." thành phần nguyên/phần thập phân (nếu có)
    pass

# assert exercise_vn_decimal("1234567.891") == "1.234.567,891"
# assert exercise_vn_decimal("1000") == "1.000"
# assert exercise_vn_decimal("0.5") == "0,5"

### Bài 6 — Mini challenge: số có dấu
Mở rộng cho số nguyên **âm**: `-1234567 → "-1.234.567"`. Viết hàm tổng quát và tự nghĩ các test biên (`-1`, `-1000`, `0`).

In [28]:
def exercise_signed_separator(n: int) -> str:
    # TODO: tách dấu ra trước, xử lý trị tuyệt đối, ghép dấu lại
    pass

# assert exercise_signed_separator(-1234567) == "-1.234.567"
# assert exercise_signed_separator(-1) == "-1"
# assert exercise_signed_separator(0) == "0"

## 16. Gợi ý 3 mức

**Bài 1:** (1) Đếm chữ số. (2) `8 % 3 = 2`. (3) Nhóm đầu 2 chữ số, sau đó hai nhóm 3.

**Bài 2:** (1) `divmod` = (thương, dư). (2) `03d` đệm 0. (3) `:,` dùng dấu phẩy.

**Bài 3:** (1) Nhóm đầu đặc biệt, các nhóm sau đều 3. (2) `range(first, len(s), 3)`. (3) Xem mục 5.

**Bài 4:** (1) `while n >= 1000`. (2) `groups.append(f"{r:03d}")`. (3) Nhóm cuối cùng `str(n)` rồi đảo ngược.

**Bài 5:** (1) `t.split(".")` cho 1 hoặc 2 phần. (2) Dùng lại hàm bài 3 cho phần nguyên `int(...)`. (3) Có phần thập phân thì nối `"," + frac`.

**Bài 6:** (1) `sign = "-" if n < 0 else ""`. (2) Xử lý `abs(n)`. (3) `sign + exercise_thousand_separator(abs(n))`.

## 17. Đáp án tham khảo

**Bài 1:** `n = 20230101`, `len = 8`, `first = 2` → nhóm `"20"`, `"230"`, `"101"` → `"20.230.101"`.

**Bài 2:** `(7, 5) 005 7,005`.

In [29]:
def answer_thousand_separator(n):
    s = str(n)
    first = len(s) % 3 or 3
    return ".".join([s[:first]] + [s[i:i + 3] for i in range(first, len(s), 3)])

def answer_separator_math(n):
    groups = []
    while n >= 1000:
        n, r = divmod(n, 1000)
        groups.append(f"{r:03d}")
    groups.append(str(n))
    return ".".join(reversed(groups))

def answer_vn_decimal(t):
    parts = t.split(".")
    head = answer_thousand_separator(int(parts[0]))
    return head + ("," + parts[1] if len(parts) == 2 else "")

def answer_signed_separator(n):
    sign = "-" if n < 0 else ""
    return sign + answer_thousand_separator(abs(n))

print("Bài 2:", divmod(7005, 1000), f"{5:03d}", f"{7005:,}")
assert answer_thousand_separator(20230101) == "20.230.101"
assert answer_thousand_separator(987) == "987"
assert answer_thousand_separator(1234) == "1.234"
assert answer_thousand_separator(1000005) == "1.000.005"
assert answer_separator_math(0) == "0"
assert answer_separator_math(1000005) == "1.000.005"
assert answer_vn_decimal("1234567.891") == "1.234.567,891"
assert answer_vn_decimal("1000") == "1.000"
assert answer_vn_decimal("0.5") == "0,5"
assert answer_signed_separator(-1234567) == "-1.234.567"
assert answer_signed_separator(-1) == "-1"
assert answer_signed_separator(-1000) == "-1.000"
assert answer_signed_separator(0) == "0"
for n in inputs[:500]:
    assert answer_separator_math(n) == answer_thousand_separator(n) == sep_builtin(n)
    assert answer_signed_separator(-n) == (f"{-n:,}".replace(",", "."))
print("Tất cả đáp án PASS")

Bài 2: (7, 5) 005 7,005
Tất cả đáp án PASS


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def thousandSeparator(self, n: int) -> str:
        s = str(n)
        first = len(s) % 3 or 3                      # nhóm đầu 1..3 chữ số
        return ".".join([s[:first]] + [s[i:i + 3] for i in range(first, len(s), 3)])
```

**Câu thần chú:** *"Nhóm từ phải: nhóm đầu dài len % 3 (hoặc 3), còn lại đều 3; làm bằng toán thì nhớ đệm 03d."*

**Checklist trước khi submit**
- [ ] Nhóm tính **từ phải**.
- [ ] Không có dấu chấm ở đầu khi số chữ số chia hết cho 3 (`987`, `123456`).
- [ ] Giữ số 0 trong nhóm giữa (`1.000.005`).
- [ ] `n = 0` trả `"0"`.
- [ ] Dấu **chấm**, không phải dấu phẩy; trả về `str`.
- [ ] Đúng tên method `thousandSeparator`.